# LeetCode 1169 — Invalid Transactions bằng Python

Link đề: https://leetcode.com/problems/invalid-transactions/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Sorting

Skeleton LeetCode:

```python
class Solution:
    def invalidTransactions(self, transactions: List[str]) -> List[str]:
        ...
```

Ghi chú: đáp án là **danh sách chuỗi** — các giao dịch "có thể không hợp lệ", **thứ tự tuỳ ý**. Một giao dịch không hợp lệ nếu số tiền `> 1000`, **hoặc** có một giao dịch khác **cùng tên**, **khác thành phố**, cách nhau **không quá 60 phút** (tính cả 60). Lưu ý: có thể có **chuỗi giao dịch trùng nhau hoàn toàn** — mỗi bản được xét riêng theo chỉ số. Vì đáp án nhiều thứ tự, test bằng cách **so sánh sau khi sort**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai điều kiện không hợp lệ độc lập (tiền và xung đột thời gian–thành phố), "trong vòng 60 phút" là `|t1 - t2| <= 60`.
2. Nắm công cụ: `str.split(',')`, ép kiểu `int`, `collections.defaultdict` để **nhóm theo tên**, sort theo khoá, cửa sổ trượt với `Counter`.
3. Đi từ **so sánh mọi cặp** → **nhóm theo tên rồi so cặp trong nhóm** → **nhóm + sort + cửa sổ trượt** → **bảng thời gian theo tên** (tận dụng `time ≤ 1000`).
4. Biết làm việc với **chỉ số** thay vì chuỗi để xử lý giao dịch trùng lặp.
5. Tránh lỗi: dùng `set` chuỗi làm mất bản trùng, `< 60` thay vì `<= 60`, `>= 1000` thay vì `> 1000`, chỉ đánh dấu một bên của cặp xung đột.

## 2. Đề bài & ví dụ

Một giao dịch **có thể không hợp lệ** nếu:

- số tiền **vượt quá** `$1000`, hoặc
- nó xảy ra **trong vòng (kể cả) 60 phút** so với **một giao dịch khác** có **cùng tên** nhưng ở **thành phố khác**.

Mỗi `transactions[i]` có dạng `"{name},{time},{amount},{city}"` (time tính bằng phút). Trả về danh sách các giao dịch có thể không hợp lệ, **thứ tự bất kỳ**.

**Ví dụ 1:** `["alice,20,800,mtv","alice,50,100,beijing"]` → cả hai (cách 30 phút, khác thành phố).

**Ví dụ 2:** `["alice,20,800,mtv","alice,50,1200,mtv"]` → `["alice,50,1200,mtv"]` (cùng thành phố; chỉ cái thứ hai vượt 1000).

**Ví dụ 3:** `["alice,20,800,mtv","bob,50,1200,mtv"]` → `["bob,50,1200,mtv"]`.

**Ràng buộc:** `transactions.length <= 1000`; `name`, `city` là chữ thường dài 1–10; `time` trong `[0, 1000]`; `amount` trong `[0, 2000]`.

In [1]:
import random
import time
from bisect import bisect_left, bisect_right
from collections import Counter, defaultdict
from typing import List

examples = [
    (["alice,20,800,mtv", "alice,50,100,beijing"], ["alice,20,800,mtv", "alice,50,100,beijing"]),
    (["alice,20,800,mtv", "alice,50,1200,mtv"], ["alice,50,1200,mtv"]),
    (["alice,20,800,mtv", "bob,50,1200,mtv"], ["bob,50,1200,mtv"]),
]


def same(a, b):
    """So sánh hai danh sách không quan tâm thứ tự (giữ cả bản trùng)."""
    return sorted(a) == sorted(b)

for tx, want in examples:
    print(tx, "->", want)

['alice,20,800,mtv', 'alice,50,100,beijing'] -> ['alice,20,800,mtv', 'alice,50,100,beijing']
['alice,20,800,mtv', 'alice,50,1200,mtv'] -> ['alice,50,1200,mtv']
['alice,20,800,mtv', 'bob,50,1200,mtv'] -> ['bob,50,1200,mtv']


## 3. Kiến thức nền

### 3.1 Tách chuỗi bằng `split` và ép kiểu

`"alice,20,800,mtv".split(",")` → `['alice', '20', '800', 'mtv']`. `time`, `amount` là **chuỗi** — phải `int(...)` trước khi so sánh, nếu không `"900" > "1000"` (so sánh từ điển) sẽ cho `True`!

In [2]:
name, t, amt, city = "alice,20,800,mtv".split(",")
print(name, int(t), int(amt), city)
print('"900" > "1000" ?', "900" > "1000", "| 900 > 1000 ?", 900 > 1000)
assert ("900" > "1000") is True and (900 > 1000) is False

alice 20 800 mtv
"900" > "1000" ? True | 900 > 1000 ? False


### 3.2 Parse thành tuple có chỉ số

Ta chuyển mỗi giao dịch thành `(name, time, amount, city, index)`. Giữ `index` để đánh dấu không hợp lệ theo **vị trí**, tránh nhầm các chuỗi trùng nhau.

In [3]:
def parse(transactions):
    out = []
    for i, s in enumerate(transactions):
        name, t, amt, city = s.split(",")
        out.append((name, int(t), int(amt), city, i))
    return out

print(parse(examples[0][0]))

[('alice', 20, 800, 'mtv', 0), ('alice', 50, 100, 'beijing', 1)]


### 3.3 `defaultdict(list)` — nhóm theo khoá

`groups[name].append(x)` tự tạo list rỗng khi khoá mới xuất hiện. Hai giao dịch chỉ có thể xung đột khi **cùng tên**, nên nhóm theo tên giúp giảm số cặp cần xét.

In [4]:
groups = defaultdict(list)
for rec in parse(["alice,20,800,mtv", "bob,50,1200,mtv", "alice,70,10,x"]):
    groups[rec[0]].append(rec)
print(dict(groups))

{'alice': [('alice', 20, 800, 'mtv', 0), ('alice', 70, 10, 'x', 2)], 'bob': [('bob', 50, 1200, 'mtv', 1)]}


### 3.4 Sort theo khoá và `bisect`

`lst.sort(key=lambda r: r[1])` sort theo thời gian. Với mảng thời gian đã sort, `bisect_left(times, t - 60)` và `bisect_right(times, t + 60)` cho đoạn chỉ số các giao dịch trong `[t-60, t+60]`.

In [5]:
times = [10, 20, 75, 80, 200]
t = 20
lo, hi = bisect_left(times, t - 60), bisect_right(times, t + 60)
print(times[lo:hi])
assert times[lo:hi] == [10, 20, 75, 80]

[10, 20, 75, 80]


### 3.5 `Counter` làm "túi" đếm trong cửa sổ trượt

Ta có thể thêm/bớt phần tử khỏi `Counter` và biết **số phần tử phân biệt** (số khoá có đếm > 0). Khi bớt về 0, phải `del` khoá để `len(counter)` vẫn đúng.

In [6]:
bag = Counter()
for c in ["mtv", "mtv", "sf"]:
    bag[c] += 1
print(len(bag), bag)
bag["sf"] -= 1
if bag["sf"] == 0:
    del bag["sf"]
print(len(bag), bag)
assert len(bag) == 1

2 Counter({'mtv': 2, 'sf': 1})
1 Counter({'mtv': 2})


## 4. Ý tưởng nền tảng / trực giác

Tính tay `["alice,20,800,mtv", "alice,50,100,beijing", "alice,51,100,frankfurt", "bob,50,1200,mtv"]`:

- `bob,50,1200` → tiền `1200 > 1000` → **không hợp lệ**.
- Nhóm `alice`: thời gian `20 (mtv)`, `50 (beijing)`, `51 (frankfurt)`.
  - `20` vs `50`: cách 30, khác thành phố → cả hai **không hợp lệ**.
  - `20` vs `51`: cách 31, khác thành phố → `51` cũng không hợp lệ.
- Đáp án: cả 4 giao dịch.

Quan sát: điều kiện xung đột là **đối xứng** — nếu A xung đột với B thì B cũng xung đột với A. Và chỉ cần **một** giao dịch xung đột là đủ để đánh dấu.

In [7]:
demo = ["alice,20,800,mtv", "alice,50,100,beijing", "alice,51,100,frankfurt", "bob,50,1200,mtv"]
for r in parse(demo):
    print(r)

('alice', 20, 800, 'mtv', 0)
('alice', 50, 100, 'beijing', 1)
('alice', 51, 100, 'frankfurt', 2)
('bob', 50, 1200, 'mtv', 3)


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** với mỗi giao dịch `i`, kiểm tra tiền; rồi so với **mọi** `j ≠ i`: cùng tên, khác thành phố, `|t_i − t_j| ≤ 60`.

In [8]:
def invalid_brute(transactions):
    recs = parse(transactions)
    res = []
    for name, t, amt, city, i in recs:
        bad = amt > 1000
        if not bad:
            for name2, t2, _, city2, j in recs:
                if j != i and name2 == name and city2 != city and abs(t - t2) <= 60:
                    bad = True
                    break
        if bad:
            res.append(transactions[i])
    return res

for tx, want in examples:
    got = invalid_brute(tx)
    print(got)
    assert same(got, want)

['alice,20,800,mtv', 'alice,50,100,beijing']
['alice,50,1200,mtv']
['bob,50,1200,mtv']


**Độ phức tạp:** `O(n²)` so sánh (mỗi so sánh `O(1)` sau khi parse), `O(n)` bộ nhớ. Với `n ≤ 1000` đây là `10^6` — **thực ra đủ nhanh** để nộp! Nhưng nếu `n` lên `10^5` thì không ổn.

**Nút thắt:** so sánh cả các cặp khác tên (vô ích) và các cặp cách xa nhau về thời gian.

## 6. Bước đột phá

> Xung đột chỉ xảy ra giữa giao dịch **cùng tên** và **gần nhau về thời gian**. Nhóm theo tên, sort theo thời gian trong nhóm → các ứng viên xung đột của một giao dịch nằm trong một **đoạn liên tiếp** `[t − 60, t + 60]`.

Invariant: với giao dịch `x` trong nhóm đã sort, `x` xung đột ⟺ đoạn thời gian `[t_x − 60, t_x + 60]` của nhóm chứa **ít nhất hai thành phố khác nhau** (vì chính `x` nằm trong đoạn với thành phố của nó; có thành phố thứ hai nghĩa là có giao dịch khác thành phố).

Quy tắc:

- Nếu `amount > 1000` → không hợp lệ.
- Nếu cửa sổ `[t − 60, t + 60]` của cùng tên có `≥ 2` thành phố phân biệt → không hợp lệ.

Khi duyệt `x` theo thời gian tăng dần, cả hai đầu cửa sổ đều **chỉ tiến lên** → hai con trỏ, tổng `O(k)` cho nhóm kích thước `k`.

In [9]:
# Minh hoạ: cửa sổ ±60 của từng giao dịch trong nhóm alice
alice = sorted([r for r in parse(demo) if r[0] == "alice"], key=lambda r: r[1])
for r in alice:
    window = [x[3] for x in alice if abs(x[1] - r[1]) <= 60]
    print(r[1], r[3], "-> thành phố trong cửa sổ:", window, "| phân biệt:", len(set(window)))

20 mtv -> thành phố trong cửa sổ: ['mtv', 'beijing', 'frankfurt'] | phân biệt: 3
50 beijing -> thành phố trong cửa sổ: ['mtv', 'beijing', 'frankfurt'] | phân biệt: 3
51 frankfurt -> thành phố trong cửa sổ: ['mtv', 'beijing', 'frankfurt'] | phân biệt: 3


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Nhóm theo tên, so cặp trong nhóm

Giảm hằng số: chỉ so các cặp cùng tên. Đánh dấu cả hai phía khi tìm thấy xung đột (xét mỗi cặp một lần).

In [10]:
def invalid_group_pairs(transactions):
    recs = parse(transactions)
    bad = [r[2] > 1000 for r in recs]
    groups = defaultdict(list)
    for r in recs:
        groups[r[0]].append(r)
    for lst in groups.values():
        for a in range(len(lst)):
            for b in range(a + 1, len(lst)):
                _, t1, _, c1, i = lst[a]
                _, t2, _, c2, j = lst[b]
                if c1 != c2 and abs(t1 - t2) <= 60:
                    bad[i] = bad[j] = True
    return [transactions[i] for i in range(len(recs)) if bad[i]]

for tx, want in examples:
    assert same(invalid_group_pairs(tx), want)
print("group pairs OK")

group pairs OK


**Độ phức tạp:** `O(Σ k_g²)` với `k_g` = số giao dịch của tên `g`; tệ nhất (tất cả cùng tên) vẫn `O(n²)`. Bộ nhớ `O(n)`.

### 7.2 Cách 3 — Nhóm + sort + cửa sổ trượt hai con trỏ

Với mỗi nhóm đã sort theo thời gian, duy trì cửa sổ `[left, right)` sao cho chứa đúng các giao dịch có `t ∈ [t_x − 60, t_x + 60]`, cùng `Counter` thành phố.

In [11]:
def invalid_window(transactions):
    recs = parse(transactions)
    bad = [r[2] > 1000 for r in recs]
    groups = defaultdict(list)
    for r in recs:
        groups[r[0]].append(r)
    for lst in groups.values():
        lst.sort(key=lambda r: r[1])
        bag = Counter()
        left = right = 0
        k = len(lst)
        for _, t, _, _, i in lst:
            while right < k and lst[right][1] <= t + 60:     # mở rộng phải
                bag[lst[right][3]] += 1
                right += 1
            while lst[left][1] < t - 60:                     # thu hẹp trái
                c = lst[left][3]
                bag[c] -= 1
                if bag[c] == 0:
                    del bag[c]
                left += 1
            if len(bag) >= 2:
                bad[i] = True
    return [transactions[i] for i in range(len(recs)) if bad[i]]

for tx, want in examples:
    assert same(invalid_window(tx), want)
print("window OK")

window OK


**Độ phức tạp:** sort `O(n log n)`, cửa sổ `O(n)` tổng → `O(n log n)`. Bộ nhớ `O(n)`.

Vì sao `while lst[left][1] < t - 60` không vượt mảng? Phần tử `x` hiện tại có `t ≥ t − 60`, và `left ≤` vị trí của `x` (vì `x` đã nằm trong cửa sổ sau bước mở rộng phải), nên vòng lặp luôn dừng trước khi qua `x`.

### 7.3 Cách 4 — Bảng thời gian theo tên (tận dụng `time ≤ 1000`)

`seen[name][t]` = tập thành phố có giao dịch tên `name` tại phút `t`. Với mỗi giao dịch, quét `t − 60 … t + 60` (121 phút) xem có thành phố khác. `O(121 · n)` — tuyến tính theo `n`, không cần sort.

In [12]:
def invalid_time_table(transactions):
    recs = parse(transactions)
    seen = defaultdict(lambda: defaultdict(set))
    for name, t, _, city, _ in recs:
        seen[name][t].add(city)
    res = []
    for name, t, amt, city, i in recs:
        bad = amt > 1000
        if not bad:
            table = seen[name]
            for tt in range(t - 60, t + 61):
                cities = table.get(tt)
                if cities and (len(cities) > 1 or city not in cities):
                    bad = True
                    break
        if bad:
            res.append(transactions[i])
    return res

for tx, want in examples:
    assert same(invalid_time_table(tx), want)
print("time table OK")

time table OK


Điều kiện `len(cities) > 1 or city not in cities`: tại phút `tt` có một thành phố **khác** `city` khi và chỉ khi tập có ≥ 2 phần tử (chắc chắn có cái khác `city`) hoặc tập 1 phần tử mà phần tử đó không phải `city`.

**Độ phức tạp:** `O(121 · n)` thời gian, `O(n)` bộ nhớ.

## 8. TRACE từng bước

Trace cách 3 cho nhóm `alice` gồm `(10, a), (40, a), (90, b), (200, b), (230, c)`:

| x (t, city) | right sau mở rộng | left sau thu hẹp | cửa sổ | #thành phố | bad? |
|-------------|-------------------|------------------|--------|------------|------|
| (10,a) | 2 (t ≤ 70) | 0 | 10a 40a | 1 | không |
| (40,a) | 3 (t ≤ 100) | 0 | 10a 40a 90b | 2 | **có** |
| (90,b) | 3 (t ≤ 150) | 1 (t ≥ 30) | 40a 90b | 2 | **có** |
| (200,b) | 5 (t ≤ 260) | 3 (t ≥ 140) | 200b 230c | 2 | **có** |
| (230,c) | 5 | 3 (t ≥ 170) | 200b 230c | 2 | **có** |

In [13]:
def trace_window(transactions):
    recs = parse(transactions)
    groups = defaultdict(list)
    for r in recs:
        groups[r[0]].append(r)
    for name, lst in groups.items():
        lst.sort(key=lambda r: r[1])
        print("nhóm", name, [(r[1], r[3]) for r in lst])
        bag, left, right = Counter(), 0, 0
        for _, t, _, city, i in lst:
            while right < len(lst) and lst[right][1] <= t + 60:
                bag[lst[right][3]] += 1; right += 1
            while lst[left][1] < t - 60:
                c = lst[left][3]; bag[c] -= 1
                if bag[c] == 0: del bag[c]
                left += 1
            win = [(r[1], r[3]) for r in lst[left:right]]
            print(f"  x=({t},{city}) left={left} right={right} window={win} cities={dict(bag)} bad={len(bag) >= 2}")

trace_window(["alice,10,1,a", "alice,40,1,a", "alice,90,1,b", "alice,200,1,b", "alice,230,1,c", "bob,5,1,z"])

nhóm alice [(10, 'a'), (40, 'a'), (90, 'b'), (200, 'b'), (230, 'c')]
  x=(10,a) left=0 right=2 window=[(10, 'a'), (40, 'a')] cities={'a': 2} bad=False
  x=(40,a) left=0 right=3 window=[(10, 'a'), (40, 'a'), (90, 'b')] cities={'a': 2, 'b': 1} bad=True
  x=(90,b) left=1 right=3 window=[(40, 'a'), (90, 'b')] cities={'a': 1, 'b': 1} bad=True
  x=(200,b) left=3 right=5 window=[(200, 'b'), (230, 'c')] cities={'b': 1, 'c': 1} bad=True
  x=(230,c) left=3 right=5 window=[(200, 'b'), (230, 'c')] cities={'b': 1, 'c': 1} bad=True
nhóm bob [(5, 'z')]
  x=(5,z) left=0 right=1 window=[(5, 'z')] cities={'z': 1} bad=False


## 9. Code nộp LeetCode

Chọn cách 3 (nhóm + sort + cửa sổ trượt) — `O(n log n)`, mở rộng tốt.

In [14]:
from collections import Counter, defaultdict
from typing import List


class Solution:
    def invalidTransactions(self, transactions: List[str]) -> List[str]:
        recs = []
        for i, s in enumerate(transactions):
            name, t, amt, city = s.split(",")
            recs.append((name, int(t), int(amt), city, i))
        bad = [r[2] > 1000 for r in recs]

        groups = defaultdict(list)
        for r in recs:
            groups[r[0]].append(r)

        for lst in groups.values():
            lst.sort(key=lambda r: r[1])
            bag = Counter()
            left = right = 0
            for _, t, _, _, i in lst:
                while right < len(lst) and lst[right][1] <= t + 60:
                    bag[lst[right][3]] += 1
                    right += 1
                while lst[left][1] < t - 60:
                    c = lst[left][3]
                    bag[c] -= 1
                    if bag[c] == 0:
                        del bag[c]
                    left += 1
                if len(bag) >= 2:          # có thành phố khác trong vòng ±60 phút
                    bad[i] = True

        return [transactions[i] for i in range(len(transactions)) if bad[i]]


sol = Solution()
for tx, want in examples:
    assert same(sol.invalidTransactions(tx), want)
print("Solution OK")

Solution OK


## 10. Vì sao đúng

**Điều kiện tiền:** kiểm tra trực tiếp `amount > 1000` cho từng chỉ số.

**Điều kiện xung đột:** cố định tên `g` và nhóm đã sort theo thời gian. Với giao dịch `x` thời gian `t`:

- Sau vòng mở rộng phải, `right` là chỉ số đầu tiên có thời gian `> t + 60` (vì `t` không giảm theo thứ tự duyệt, `right` chỉ tiến).
- Sau vòng thu hẹp trái, `left` là chỉ số đầu tiên có thời gian `≥ t − 60` (tương tự, `t − 60` không giảm).
- Vậy cửa sổ = **đúng** tập giao dịch cùng tên có `|t' − t| ≤ 60`, bao gồm `x`; `bag` đếm thành phố của chúng.
- `x` xung đột ⟺ tồn tại `y` trong cửa sổ có `city_y ≠ city_x` ⟺ cửa sổ có ≥ 2 thành phố phân biệt (vì `city_x` chắc chắn có mặt).

**Trùng lặp:** ta làm việc theo chỉ số `i`, xuất `transactions[i]` cho mọi `i` bị đánh dấu → các bản trùng được xuất đúng số lần. Hai bản trùng hệt nhau không xung đột với nhau (cùng thành phố) — đúng với đặc tả.

## 11. Time / Space Complexity

Gọi `n` = số giao dịch, `L ≤ 33` = độ dài chuỗi (parse `O(L)`, coi là hằng).

| Cách | Time | Space |
|------|------|-------|
| 1. Brute mọi cặp | `O(n²)` | `O(n)` |
| 2. Nhóm + cặp trong nhóm | `O(Σ k_g²)` ≤ `O(n²)` | `O(n)` |
| 3. Nhóm + sort + cửa sổ | `O(n log n)` | `O(n)` |
| 4. Bảng thời gian | `O(121 · n)` | `O(n)` |

Với `n ≤ 1000` mọi cách đều qua; cách 3/4 thể hiện tư duy mở rộng được.

## 12. Bộ test

### 12.1 Ví dụ + edge case

In [15]:
impls = {
    "brute": invalid_brute,
    "group_pairs": invalid_group_pairs,
    "window": invalid_window,
    "time_table": invalid_time_table,
    "Solution": Solution().invalidTransactions,
}
edge = [
    ([], []),
    (["a,0,1000,x"], []),                                  # đúng 1000 không vượt
    (["a,0,1001,x"], ["a,0,1001,x"]),
    (["a,0,1,x", "a,60,1,y"], ["a,0,1,x", "a,60,1,y"]),    # đúng 60 phút → tính
    (["a,0,1,x", "a,61,1,y"], []),                         # 61 phút → không
    (["a,10,1,x", "a,10,1,x"], []),                        # trùng hệt, cùng thành phố
    (["a,10,1,x", "a,10,1,x", "a,20,1,y"], ["a,10,1,x", "a,10,1,x", "a,20,1,y"]),  # trùng phải xuất đủ
    (["a,10,1,x", "b,20,1,y"], []),                        # khác tên
    (["a,0,1,x", "a,50,1,x", "a,100,1,y"], ["a,50,1,x", "a,100,1,y"]),
]
for name, f in impls.items():
    for tx, want in examples + edge:
        assert same(f(tx), want), (name, tx, f(tx))
print("all impls pass", len(examples + edge))

all impls pass 12


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

Tên trong `{a, b}`, thành phố trong `{x, y, z}`, thời gian nhỏ để hay rơi vào biên 60, tiền quanh 1000, thỉnh thoảng nhân đôi một giao dịch.

In [16]:
def random_tx(rng, n):
    out = []
    for _ in range(n):
        if out and rng.random() < 0.1:
            out.append(rng.choice(out))          # bản trùng
            continue
        out.append(f"{rng.choice('ab')},{rng.randint(0, 200)},{rng.choice([0, 999, 1000, 1001, 2000])},{rng.choice('xyz')}")
    return out

rng = random.Random(1169)
for _ in range(500):
    tx = random_tx(rng, rng.randint(0, 15))
    want = invalid_brute(tx)
    for name, f in impls.items():
        assert same(f(tx), want), (name, tx)
print("random cross-check OK")

random cross-check OK


### 12.3 Test giới hạn

`n = 1000` (giới hạn đề) và `n = 20000` (gấp 20 lần, để thấy khác biệt `O(n²)` vs `O(n log n)`).

In [17]:
for n in (1000, 20000):
    rng = random.Random(n)
    tx = [f"{rng.choice(['alice', 'bob', 'carol'])},{rng.randint(0, 1000)},{rng.randint(0, 2000)},{rng.choice(['mtv', 'sf', 'ny'])}" for _ in range(n)]
    names = ["window", "time_table", "Solution"] + (["brute", "group_pairs"] if n <= 1000 else [])
    results = {}
    for name in names:
        t0 = time.perf_counter()
        results[name] = sorted(impls[name](tx))
        print(f"n={n} {name:12} {len(results[name])} invalid  {time.perf_counter() - t0:.3f}s")
    assert len({tuple(v) for v in results.values()}) == 1

n=1000 window       1000 invalid  0.002s
n=1000 time_table   1000 invalid  0.001s
n=1000 Solution     1000 invalid  0.001s
n=1000 brute        1000 invalid  0.002s


n=1000 group_pairs  1000 invalid  0.023s


n=20000 window       20000 invalid  0.046s


n=20000 time_table   20000 invalid  0.031s
n=20000 Solution     20000 invalid  0.033s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `set` các chuỗi kết quả (mất bản trùng)

In [18]:
def wrong_set_result(transactions):
    res = set()
    recs = parse(transactions)
    for name, t, amt, city, i in recs:
        if amt > 1000 or any(n2 == name and c2 != city and abs(t - t2) <= 60 for n2, t2, _, c2, _ in recs):
            res.add(transactions[i])
    return list(res)

tx = ["a,10,1,x", "a,10,1,x", "a,20,1,y"]
print("sai:", wrong_set_result(tx), "| đúng:", invalid_brute(tx))
assert not same(wrong_set_result(tx), invalid_brute(tx))

sai: ['a,20,1,y', 'a,10,1,x'] | đúng: ['a,10,1,x', 'a,10,1,x', 'a,20,1,y']


**Vì sao sai:** hai giao dịch `"a,10,1,x"` là **hai giao dịch khác nhau** (khác chỉ số), cả hai đều không hợp lệ; `set` gộp làm một. **Sửa:** đánh dấu theo chỉ số, xuất list.

### Lỗi 2 — So sánh `< 60` (bỏ biên)

In [19]:
def wrong_strict(transactions):
    recs = parse(transactions)
    return [transactions[i] for name, t, amt, city, i in recs
            if amt > 1000 or any(n2 == name and c2 != city and abs(t - t2) < 60 for n2, t2, _, c2, _ in recs)]

tx = ["a,0,1,x", "a,60,1,y"]
print("sai:", wrong_strict(tx), "| đúng:", invalid_brute(tx))
assert not same(wrong_strict(tx), invalid_brute(tx))

sai: [] | đúng: ['a,0,1,x', 'a,60,1,y']


**Vì sao sai:** đề nói "within (and including) 60 minutes" → `<= 60`. **Sửa:** `abs(t1 - t2) <= 60`.

### Lỗi 3 — So sánh tiền dạng chuỗi

In [20]:
def wrong_string_amount(transactions):
    res = []
    for s in transactions:
        name, t, amt, city = s.split(",")
        if amt > "1000":                       # so sánh từ điển!
            res.append(s)
    return res

tx = ["a,0,999,x", "b,0,1200,y"]
print("sai:", wrong_string_amount(tx), "| đúng:", invalid_brute(tx))
assert not same(wrong_string_amount(tx), invalid_brute(tx))

sai: ['a,0,999,x', 'b,0,1200,y'] | đúng: ['b,0,1200,y']


**Vì sao sai:** `"999" > "1000"` vì `'9' > '1'`; còn `"1200" > "1000"` đúng tình cờ. **Sửa:** `int(amt) > 1000`.

### Lỗi 4 — Chỉ đánh dấu giao dịch **sau** trong cặp xung đột

In [21]:
def wrong_one_side(transactions):
    recs = sorted(parse(transactions), key=lambda r: r[1])
    bad = [r[2] > 1000 for r in parse(transactions)]
    for a in range(len(recs)):
        for b in range(a):                     # chỉ nhìn về quá khứ
            if recs[a][0] == recs[b][0] and recs[a][3] != recs[b][3] and recs[a][1] - recs[b][1] <= 60:
                bad[recs[a][4]] = True         # quên đánh dấu recs[b]
    return [transactions[i] for i in range(len(transactions)) if bad[i]]

tx = examples[0][0]
print("sai:", wrong_one_side(tx), "| đúng:", invalid_brute(tx))
assert not same(wrong_one_side(tx), invalid_brute(tx))

sai: ['alice,50,100,beijing'] | đúng: ['alice,20,800,mtv', 'alice,50,100,beijing']


**Vì sao sai:** quan hệ xung đột **đối xứng**; giao dịch sớm hơn cũng không hợp lệ. **Sửa:** đánh dấu cả hai (`bad[i] = bad[j] = True`) hoặc xét cửa sổ hai phía `[t − 60, t + 60]`.

### Lỗi 5 — Cửa sổ chỉ nhìn một phía `[t − 60, t]`

Tương tự lỗi 4 nhưng ở dạng cửa sổ trượt: nếu chỉ thu hẹp trái mà không mở rộng phải tới `t + 60`, giao dịch sớm hơn bị bỏ sót. Cách 3 mở rộng `right` tới `t + 60` **trước** khi kiểm tra.

## 14. Mẹo & nhận dạng pattern

- **Dữ liệu dạng chuỗi CSV** → parse một lần thành tuple có kiểu đúng (`int`), giữ **chỉ số gốc**.
- **"Cùng khoá X và gần nhau theo Y"** → nhóm theo X (`defaultdict(list)`), sort theo Y, cửa sổ trượt / `bisect`.
- **Miền giá trị nhỏ** (time ≤ 1000) → có thể dùng bảng theo giá trị thay vì sort.
- **"Có phần tử khác loại trong cửa sổ"** → đếm số loại phân biệt trong cửa sổ bằng `Counter` + `del` khi về 0.
- Đáp án "any order" + có thể trùng → test bằng `sorted(a) == sorted(b)`, không dùng `set`.

## 15. Biến thể

1. Ngưỡng thời gian `T` và ngưỡng tiền `A` là tham số.
2. Trả về **chỉ số** thay vì chuỗi.
3. Xung đột khi **cùng thành phố** quá `k` giao dịch trong 60 phút (phát hiện spam) → cửa sổ trượt đếm số phần tử.
4. Dữ liệu đến dạng **luồng** (stream) theo thời gian tăng dần — chỉ cần giữ cửa sổ 60 phút gần nhất, nhưng giao dịch cũ có thể trở thành không hợp lệ khi giao dịch mới tới.
5. **LC 1604 Alert Using Same Key-Card Three or More Times in a One Hour Period** — nhóm theo tên, sort thời gian, cửa sổ.

In [22]:
# Biến thể 1 + 2: tham số hoá và trả về chỉ số
def invalid_indices(transactions, T=60, A=1000):
    recs = parse(transactions)
    groups = defaultdict(list)
    for r in recs:
        groups[r[0]].append(r)
    bad = [r[2] > A for r in recs]
    for lst in groups.values():
        lst.sort(key=lambda r: r[1])
        times = [r[1] for r in lst]
        for _, t, _, city, i in lst:
            lo, hi = bisect_left(times, t - T), bisect_right(times, t + T)
            if any(lst[k][3] != city for k in range(lo, hi)):
                bad[i] = True
    return [i for i, b in enumerate(bad) if b]

print(invalid_indices(examples[0][0]), invalid_indices(["a,0,1,x", "a,100,1,y"], T=100))
assert invalid_indices(["a,0,1,x", "a,100,1,y"], T=100) == [0, 1]

[0, 1] [0, 1]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Brute | So mọi cặp | `O(n²)` | `O(n)` | `n ≤ 1000` (đề này), kiểm chứng |
| 2. Nhóm + cặp | Chỉ so cặp cùng tên | `O(Σk²)` | `O(n)` | Nhiều tên khác nhau |
| 3. Nhóm + sort + cửa sổ | Hai con trỏ ±60 | `O(n log n)` | `O(n)` | Tổng quát, `n` lớn |
| 4. Bảng thời gian | Quét 121 phút | `O(121 n)` | `O(n)` | Miền thời gian nhỏ |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `["x,0,500,a", "x,61,500,b", "x,121,500,a"]` → ?

**Bài 2 (tính tay):** với `["a,100,10,p", "a,40,10,q", "a,160,1500,p", "a,220,10,q"]`, xác định từng giao dịch hợp lệ/không và lý do.

**Bài 3 (điền TODO):** viết hàm `conflict(r1, r2)` nhận hai tuple đã parse và trả `True` nếu chúng xung đột.

**Bài 4 (tự viết lại):** viết lại cách 3 dùng `bisect` (như biến thể ở mục 15) nhưng kiểm tra "có thành phố khác" trong `O(1)` mỗi giao dịch bằng **prefix count** theo thành phố.

**Bài 5 (mini challenge):** LC 1604 — cho `keyName[i]`, `keyTime[i]` dạng `"HH:MM"`, trả về danh sách tên (sort) dùng thẻ **≥ 3 lần trong một giờ** (`t3 − t1 <= 60`).

In [23]:
# Bài 3 — TODO
def conflict(r1, r2):
    # r = (name, time, amount, city, index)
    raise NotImplementedError

# assert conflict(("a", 0, 1, "x", 0), ("a", 60, 1, "y", 1)) is True
# assert conflict(("a", 0, 1, "x", 0), ("a", 60, 1, "x", 1)) is False

In [24]:
# Bài 5 — TODO
def alert_names(keyName, keyTime):
    # TODO
    pass

# assert alert_names(["daniel","daniel","daniel","luis","luis","luis","luis"],
#                    ["10:00","10:40","11:00","09:00","11:00","13:00","15:00"]) == ["daniel"]

## 18. Gợi ý 3 mức

- **Bài 1:** Mức 1: `0` vs `61` cách 61. Mức 2: `61` vs `121` cách 60, khác thành phố. Mức 3: hai giao dịch sau không hợp lệ.
- **Bài 2:** Mức 1: sort theo thời gian `40q, 100p, 160p, 220q`. Mức 2: `40q–100p` cách 60 → cả hai; `160p–220q` cách 60 → cả hai. Mức 3: `160p` còn vượt tiền; cả 4 không hợp lệ.
- **Bài 3:** Mức 1: cùng tên. Mức 2: khác thành phố. Mức 3: `abs(Δt) <= 60`; **không** cần khác chỉ số vì cùng chỉ số thì cùng thành phố.
- **Bài 4:** Mức 1: trong nhóm, `pref[c][k]` = số giao dịch thành phố `c` trong `lst[:k]`. Mức 2: số giao dịch cùng thành phố trong `[lo, hi)` = `pref[city][hi] − pref[city][lo]`. Mức 3: có thành phố khác ⟺ con số đó `< hi − lo`.
- **Bài 5:** Mức 1: đổi `"HH:MM"` → phút. Mức 2: nhóm theo tên, sort. Mức 3: kiểm tra `t[k] − t[k−2] <= 60`.

## 19. Đáp án tham khảo

In [25]:
# Bài 1
assert same(Solution().invalidTransactions(["x,0,500,a", "x,61,500,b", "x,121,500,a"]), ["x,61,500,b", "x,121,500,a"])
# Bài 2
tx2 = ["a,100,10,p", "a,40,10,q", "a,160,1500,p", "a,220,10,q"]
assert same(Solution().invalidTransactions(tx2), tx2)

# Bài 3
def conflict_sol(r1, r2):
    return r1[0] == r2[0] and r1[3] != r2[3] and abs(r1[1] - r2[1]) <= 60

assert conflict_sol(("a", 0, 1, "x", 0), ("a", 60, 1, "y", 1)) is True
assert conflict_sol(("a", 0, 1, "x", 0), ("a", 60, 1, "x", 1)) is False

# Bài 4
def invalid_prefix(transactions):
    recs = parse(transactions)
    bad = [r[2] > 1000 for r in recs]
    groups = defaultdict(list)
    for r in recs:
        groups[r[0]].append(r)
    for lst in groups.values():
        lst.sort(key=lambda r: r[1])
        times = [r[1] for r in lst]
        pref = defaultdict(lambda: [0] * (len(lst) + 1))
        cities = {r[3] for r in lst}
        for c in cities:
            arr = pref[c]
            for k, r in enumerate(lst):
                arr[k + 1] = arr[k] + (r[3] == c)
        for _, t, _, city, i in lst:
            lo, hi = bisect_left(times, t - 60), bisect_right(times, t + 60)
            if pref[city][hi] - pref[city][lo] < hi - lo:
                bad[i] = True
    return [transactions[i] for i in range(len(recs)) if bad[i]]

rng = random.Random(4)
for _ in range(300):
    tx = random_tx(rng, rng.randint(0, 12))
    assert same(invalid_prefix(tx), invalid_brute(tx))
print("bài 1-4 OK")

bài 1-4 OK


In [26]:
# Bài 5
def alert_names_sol(keyName, keyTime):
    g = defaultdict(list)
    for name, hm in zip(keyName, keyTime):
        h, m = map(int, hm.split(":"))
        g[name].append(h * 60 + m)
    res = []
    for name, ts in g.items():
        ts.sort()
        if any(ts[k] - ts[k - 2] <= 60 for k in range(2, len(ts))):
            res.append(name)
    return sorted(res)

assert alert_names_sol(["daniel", "daniel", "daniel", "luis", "luis", "luis", "luis"],
                       ["10:00", "10:40", "11:00", "09:00", "11:00", "13:00", "15:00"]) == ["daniel"]
assert alert_names_sol(["a", "a", "a"], ["23:51", "00:10", "00:30"]) == []   # không vượt qua ngày
print("bài 5 OK")

bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
recs = [(name, int(t), int(amt), city, i) for i, (name, t, amt, city) in
        enumerate(s.split(',') for s in transactions)]
bad = [r[2] > 1000 for r in recs]
groups = defaultdict(list)
for r in recs: groups[r[0]].append(r)
for lst in groups.values():
    lst.sort(key=lambda r: r[1])
    # cửa sổ [t-60, t+60] với Counter thành phố; len(bag) >= 2 → bad
return [transactions[i] for i in range(len(recs)) if bad[i]]
```

**Checklist:**

- [ ] `int(...)` cho time và amount trước khi so sánh?
- [ ] `amount > 1000` (không phải `>=`)?
- [ ] `|Δt| <= 60` (kể cả 60)?
- [ ] Đánh dấu theo **chỉ số**, không dùng `set` chuỗi?
- [ ] Xung đột đánh dấu **cả hai** phía (cửa sổ hai chiều)?
- [ ] Chỉ so sánh giao dịch **cùng tên**?